# Synthetic retrospective consistency review

This notebook evaluates the transparent Protect/Grow heuristic against the supplied quarter-end outcome. The outcome is loaded separately and is never present in the runtime feature frame, score, UI, or LLM context.

These results describe one synthetic portfolio. They are not predictive validation and should not be interpreted as production model performance.

In [1]:
import numpy as np
import pandas as pd

from src.data import load_data, process_data
from src.scoring import score_accounts

raw = load_data()
scored = score_accounts(process_data())
assert "revenue_end_of_quarter" not in scored.columns

evaluation = scored.merge(
    raw[["account_id", "revenue_end_of_quarter"]],
    on="account_id",
    validate="one_to_one",
)
evaluation["delta"] = (
    evaluation["revenue_end_of_quarter"] - evaluation["current_revenue"]
)
evaluation["loss_dollars"] = (-evaluation["delta"]).clip(lower=0)
evaluation["growth_dollars"] = evaluation["delta"].clip(lower=0)
evaluation["change_dollars"] = evaluation["delta"].abs()

print(f"Accounts: {len(evaluation):,}")
print(evaluation["priority_action"].value_counts().to_string())

Accounts: 1,000
priority_action
Protect    903
Grow        97


In [2]:
def capture_interval(selected, dollars, samples=1000, seed=42):
    selected = np.asarray(selected, dtype=bool)
    dollars = np.asarray(dollars, dtype=float)
    rng = np.random.default_rng(seed)
    estimates = []
    for _ in range(samples):
        sample = rng.integers(0, len(dollars), len(dollars))
        denominator = dollars[sample].sum()
        estimate = dollars[sample][selected[sample]].sum() / denominator if denominator else 0
        estimates.append(estimate)
    return np.quantile(estimates, [0.025, 0.975])

rows = []
for action, value_column, outcome_column in [
    ("Protect", "protect_value", "loss_dollars"),
    ("Grow", "growth_value", "growth_dollars"),
]:
    action_accounts = evaluation[evaluation["priority_action"].eq(action)]
    selected_index = action_accounts.nlargest(50, value_column).index
    selected = evaluation.index.isin(selected_index)
    actual = evaluation[outcome_column].gt(0)
    lower, upper = capture_interval(selected, evaluation[outcome_column])
    rows.append(
        {
            "action": action,
            "assigned_accounts": len(action_accounts),
            "evaluated_top_n": int(selected.sum()),
            "outcome_precision": actual[selected].mean(),
            "outcome_recall": actual[selected].sum() / actual.sum(),
            "outcome_dollar_capture": (
                evaluation.loc[selected, outcome_column].sum()
                / evaluation[outcome_column].sum()
            ),
            "capture_95pct_low": lower,
            "capture_95pct_high": upper,
        }
    )

action_metrics = pd.DataFrame(rows).set_index("action")
action_metrics.style.format("{:.1%}", subset=[
    "outcome_precision",
    "outcome_recall",
    "outcome_dollar_capture",
    "capture_95pct_low",
    "capture_95pct_high",
])

,assigned_accounts,evaluated_top_n,outcome_precision,outcome_recall,outcome_dollar_capture,capture_95pct_low,capture_95pct_high
action,,,,,,,
Protect,903,50,16.0%,7.3%,58.4%,30.8%,76.3%
Grow,97,50,74.0%,12.4%,32.4%,20.1%,44.3%


In [3]:
ks = [10, 25, 50, 100, 200]
orders = {
    "priority_value": evaluation.sort_values("priority_value", ascending=False).index,
    "current_revenue": evaluation.sort_values("current_revenue", ascending=False).index,
    "renewal_date": evaluation.sort_values("days_to_next_renewal").index,
}

rng = np.random.default_rng(42)
rows = []
for k in ks:
    row = {"top_k": k}
    for name, order in orders.items():
        selected = evaluation.index.isin(order[:k])
        point = (
            evaluation.loc[selected, "change_dollars"].sum()
            / evaluation["change_dollars"].sum()
        )
        lower, upper = capture_interval(
            selected, evaluation["change_dollars"], seed=42 + k
        )
        row[name] = point
        row[f"{name}_95pct"] = f"{lower:.1%}–{upper:.1%}"

    random_capture = []
    for _ in range(1000):
        selected_index = rng.permutation(evaluation.index)[:k]
        random_capture.append(
            evaluation.loc[selected_index, "change_dollars"].sum()
            / evaluation["change_dollars"].sum()
        )
    row["random_mean"] = np.mean(random_capture)
    row["random_95pct"] = (
        f"{np.quantile(random_capture, 0.025):.1%}–"
        f"{np.quantile(random_capture, 0.975):.1%}"
    )
    rows.append(row)

capture_metrics = pd.DataFrame(rows).set_index("top_k")
capture_metrics.style.format("{:.1%}", subset=[
    "priority_value",
    "current_revenue",
    "renewal_date",
    "random_mean",
])

,priority_value,priority_value_95pct,current_revenue,current_revenue_95pct,renewal_date,renewal_date_95pct,random_mean,random_95pct
top_k,,,,,,,,
10,6.4%,0.7%–14.3%,2.4%,0.0%–6.5%,1.3%,0.0%–3.3%,1.0%,0.0%–4.4%
25,27.3%,13.7%–40.7%,18.7%,7.4%–31.1%,3.3%,1.0%–6.6%,2.6%,0.2%–7.6%
50,41.3%,26.8%–53.5%,34.0%,21.4%–46.0%,4.3%,1.5%–8.1%,5.0%,1.0%–10.6%
100,55.1%,43.0%–64.7%,64.9%,54.6%–73.3%,11.3%,4.8%–19.7%,10.1%,4.1%–17.0%
200,86.2%,81.4%–89.8%,87.1%,82.7%–90.1%,29.2%,17.5%–40.6%,20.2%,11.6%–29.6%


## Interpretation

- The heuristic concentrates more absolute revenue change than the simple baselines at small review budgets (10–50 accounts).
- Current revenue catches up and overtakes the heuristic at larger review budgets (100–200 accounts), so no single top-50 result should be presented as general performance.
- Grow is more precise on this synthetic portfolio; Protect captures a large share of loss dollars despite low account-level precision because revenue scale is part of the priority value.
- Bootstrap intervals quantify sampling uncertainty within this portfolio, but cannot correct synthetic-data bias or establish out-of-sample validity.